# AgroSele — Baseline Clássico (TF-IDF + Cosseno, na mão)

Esse notebook é a primeira tentativa do projeto: antes de sair usando BERTimbau
pra tudo, resolvi ver até onde um método bem antigo de PLN — TF-IDF com
similaridade de cosseno, tudo implementado do zero, sem `sklearn`, sem
embedding nenhum — consegue chegar na tarefa de **seleção de resposta** do
MilkQA.

A tarefa: dada uma pergunta de um produtor rural e 50 respostas candidatas
(1 certa + 49 erradas), o sistema tem que apontar qual é a certa.

Ideia do TF-IDF: cada palavra que aparece muito numa resposta mas é rara no
resto do corpus (respostas técnicas: nome de doença, insumo, procedimento)
vale mais peso no vetor daquele documento. A pergunta e cada candidata viram
vetores nesse espaço, e a gente ranqueia as candidatas pela similaridade de
cosseno com a pergunta.

Dataset: MilkQA (Criscuolo et al., 2018) — perguntas e respostas reais da
Embrapa Gado de Leite.

In [1]:
import csv
import math
import re
import time
from collections import Counter, defaultdict

from datasets import load_dataset

C:\Users\frede\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Tokenização e stopwords

Nada de biblioteca pronta aqui — regex simples pra separar palavras e uma
lista de stopwords do português que montei na mão (artigos, preposições,
conjugações de verbos auxiliares etc).

In [2]:
LISTA_STOPWORDS_PT = set("""
a ao aos aquela aquelas aquele aqueles aquilo as até com como da das de dele
deles depois do dos e ela elas ele eles em entre era eram essa essas esse
esses esta estamos estas estava estavam este esteja estejam estejamos estes
esteve estive estivemos estiveram estivesse estivessem estivéramos
estivéssemos estou está estávamos estão eu foi fomos for fora foram forem
formos fosse fossem fui fôramos fôssemos haja hajam hajamos havemos hei
houve houvemos houver houvera houveram houverei houverem houveremos
houveria houveriam houvermos houverá houverão houveríamos houvesse
houvessem houvéramos houvéssemos há hão isso isto já lhe lhes mais mas me
mesmo meu meus minha minhas muito na nas nem no nos nossa nossas nosso
nossos num numa não nós o os ou para pela pelas pelo pelos por qual quando
que quem se seja sejam sejamos sem serei seremos seria seriam será serão
seríamos seu seus somos sou sua suas são só também te tem temos tenha
tenham tenhamos tenho terei teremos teria teriam terá terão teríamos teu
teus teve tinha tinham tive tivemos tiver tivera tiveram tiverem tivermos
tivesse tivessem tivéramos tivéssemos tu tua tuas tá um uma você vocês
vos à às éramos é
""".split())

# regex bem simples: so letras (com acento) e numeros contam como token
padrao_token = re.compile(r"[a-zà-öø-ÿ0-9]+")


def tokenizar(texto):
    """Deixa tudo minusculo, extrai as palavras e tira as stopwords."""
    palavras = padrao_token.findall(texto.lower())
    return [p for p in palavras if p not in LISTA_STOPWORDS_PT and len(p) > 1]


# testando rapido pra ver se ta funcionando
tokenizar("O animal apresentou febre alta e perda de apetite, o que devo fazer?")

['animal', 'apresentou', 'febre', 'alta', 'perda', 'apetite', 'devo', 'fazer']

## 2. Índice TF-IDF na mão

Sem `sklearn.feature_extraction.text.TfidfVectorizer` — a ideia é mostrar o
cálculo de verdade:

- **TF** (term frequency): quantas vezes a palavra aparece no documento,
  normalizado pelo tamanho do documento.
- **IDF** (inverse document frequency): `log(N / (1 + df)) + 1`, onde `df` é
  em quantos documentos a palavra aparece. Palavra rara = IDF alto.
- **TF-IDF** = TF × IDF, e no final normalizo o vetor pra norma 1 (assim
  cosseno vira só um produto escalar).

In [3]:
class IndiceTFIDF:
    """Indice TF-IDF construido na mao sobre uma colecao de documentos."""

    def __init__(self, ids_documentos, textos_documentos):
        self.ids_documentos = list(ids_documentos)
        self.tokens_por_doc = {
            id_doc: tokenizar(texto)
            for id_doc, texto in zip(ids_documentos, textos_documentos)
        }

        # document frequency: em quantos documentos cada termo aparece
        df = defaultdict(int)
        for tokens in self.tokens_por_doc.values():
            for termo in set(tokens):
                df[termo] += 1

        n_documentos = len(self.ids_documentos)
        # IDF suavizado (evita idf=0 pra termo que aparece em todo mundo)
        self.idf = {
            termo: math.log(n_documentos / (1 + freq)) + 1.0
            for termo, freq in df.items()
        }

        # vetor TF-IDF (esparso, dict termo->peso) ja normalizado, por documento
        self.vetores = {
            id_doc: self._vetor_tfidf(tokens)
            for id_doc, tokens in self.tokens_por_doc.items()
        }

    def _vetor_tfidf(self, tokens):
        contagem = Counter(tokens)
        total = sum(contagem.values()) or 1
        vetor = {}
        for termo, qtd in contagem.items():
            tf = qtd / total
            idf = self.idf.get(termo, 0.0)
            if idf > 0:
                vetor[termo] = tf * idf
        # normaliza pra norma L2 = 1 (cosseno vira so o produto escalar)
        norma = math.sqrt(sum(peso * peso for peso in vetor.values())) or 1.0
        return {termo: peso / norma for termo, peso in vetor.items()}

    def vetorizar_pergunta(self, texto):
        """TF-IDF de um texto novo, usando o IDF que ja foi calculado do corpus."""
        return self._vetor_tfidf(tokenizar(texto))

    @staticmethod
    def cosseno(vetor_a, vetor_b):
        """Similaridade de cosseno entre dois vetores esparsos (dict).
        Como os dois ja estao normalizados, cosseno = produto escalar direto."""
        # itera pelo menor dict, e mais rapido
        if len(vetor_a) > len(vetor_b):
            vetor_a, vetor_b = vetor_b, vetor_a
        return sum(peso * vetor_b.get(termo, 0.0) for termo, peso in vetor_a.items())

## 3. Carregando os dados

Reaproveito os CSVs já exportados em `../selecao-resposta-milkqa/datasets/`
(corpus de respostas + perguntas), então não precisa baixar texto de novo —
só os metadados leves dos splits oficiais (quem é a resposta certa de cada
pergunta, quais são as 50 candidatas).

In [4]:
def carregar_csv(caminho):
    linhas = {}
    with open(caminho, encoding="utf-8") as f:
        leitor = csv.DictReader(f)
        for linha in leitor:
            linhas[linha["id"]] = linha["text"]
    return linhas


t0 = time.time()
textos_corpus = carregar_csv("../selecao-resposta-milkqa/datasets/corpus.csv")
textos_perguntas = carregar_csv("../selecao-resposta-milkqa/datasets/queries.csv")
print(f"{len(textos_corpus)} respostas | {len(textos_perguntas)} perguntas")

2657 respostas | 2657 perguntas


In [5]:
print("Construindo indice TF-IDF do corpus...")
indice = IndiceTFIDF(list(textos_corpus.keys()), list(textos_corpus.values()))
tamanho_vocabulario = len(indice.idf)
print(f"Vocabulario (depois de tirar stopword): {tamanho_vocabulario} termos unicos")
print(f"Tempo ate aqui: {time.time() - t0:.1f}s")

Construindo indice TF-IDF do corpus...


Vocabulario (depois de tirar stopword): 13627 termos unicos
Tempo ate aqui: 0.4s


In [6]:
print("Baixando/carregando os splits oficiais do MilkQA (train/dev/test)...")
ds = load_dataset("eduagarcia/MilkQA")
print(f"treino={len(ds['train'])} | dev={len(ds['dev'])} | teste={len(ds['test'])}")

Baixando/carregando os splits oficiais do MilkQA (train/dev/test)...


treino=2307 | dev=50 | teste=300


## 4. Avaliação (Accuracy@1 e MRR)

Como é tarefa de **ranqueamento** (não classificação em classes fixas), as
métricas certas são:

- **Accuracy@1**: a resposta certa ficou em 1º lugar no ranking?
- **MRR** (Mean Reciprocal Rank): média de `1/posição` da resposta certa —
  se ela fica em 3º lugar, contribui com 1/3.

In [7]:
def avaliar_ranking(indice_corpus, textos_por_id_pergunta, conjunto):
    lista_acuracia1, lista_mrr = [], []
    for linha in conjunto:
        id_pergunta = linha["query-id"]
        id_resposta_certa = linha["positive-doc-id"]
        candidatas = linha["candidates-ids"]

        vetor_pergunta = indice_corpus.vetorizar_pergunta(textos_por_id_pergunta[id_pergunta])
        pontuacoes = [
            (id_cand, IndiceTFIDF.cosseno(vetor_pergunta, indice_corpus.vetores[id_cand]))
            for id_cand in candidatas
        ]
        pontuacoes.sort(key=lambda x: -x[1])
        ids_ranqueados = [id_cand for id_cand, _ in pontuacoes]

        posicao = ids_ranqueados.index(id_resposta_certa) + 1  # 1-indexado
        lista_acuracia1.append(1.0 if posicao == 1 else 0.0)
        lista_mrr.append(1.0 / posicao)

    return sum(lista_acuracia1) / len(lista_acuracia1), sum(lista_mrr) / len(lista_mrr)

In [8]:
print("===== Avaliacao (TF-IDF + cosseno, sem treino nenhum) =====")
for nome, conjunto in [("dev", ds["dev"]), ("test", ds["test"])]:
    acuracia1, mrr = avaliar_ranking(indice, textos_perguntas, conjunto)
    print(f"  {nome:5s} ({len(conjunto):3d} perguntas) -> Accuracy@1={acuracia1:.4f}  MRR={mrr:.4f}")

n_candidatas = 50
acuracia1_aleatorio = 1 / n_candidatas
mrr_aleatorio = sum(1 / k for k in range(1, n_candidatas + 1)) / n_candidatas
print(f"\nBaseline aleatorio esperado: Accuracy@1~={acuracia1_aleatorio:.4f} MRR~={mrr_aleatorio:.4f}")
print(f"\nTempo total: {time.time() - t0:.1f}s (sem GPU, so contagem de palavras mesmo)")

===== Avaliacao (TF-IDF + cosseno, sem treino nenhum) =====
  dev   ( 50 perguntas) -> Accuracy@1=0.5600  MRR=0.6516
  test  (300 perguntas) -> Accuracy@1=0.5033  MRR=0.6102

Baseline aleatorio esperado: Accuracy@1~=0.0200 MRR~=0.0900

Tempo total: 3.8s (sem GPU, so contagem de palavras mesmo)


## Conclusão

`Accuracy@1 = 0,503` e `MRR = 0,610` no teste, em **4-5 segundos**, sem
nenhum treinamento. Pra comparação, o BERTimbau usado cru (sem nenhum ajuste,
só embedding + cosseno) fica em `Accuracy@1 = 0,277` — quase metade disso.

Isso não é coincidência: pergunta e resposta do MilkQA compartilham
vocabulário técnico bem específico (nome de doença, insumo, procedimento), e
o TF-IDF captura essa correspondência lexical **exata** muito melhor do que
um embedding genérico que não foi ajustado pra esse domínio. O BERTimbau só
passa a ganhar do TF-IDF depois que entra algum treinamento supervisionado
(ver os notebooks das outras variantes do projeto).